# AQS Eastern US Annual Data Download
## PM₂.₅ · SO₂ · NO₂ · PM10 | Eastern United States (east of 90°W)

Downloads **annual summary data** from the EPA Air Quality System (AQS) REST API for
all monitoring sites in the eastern US relevant to the aerosol radiative forcing analysis.

| Pollutant | Code | Network start | Relevance to project |
|-----------|------|---------------|----------------------|
| SO₂ | 42401 | **1971** | Sulfate precursor — full Clean Air Act era from 1970 amendments |
| PM10 | 81102 | **1987** | Bridges TSP → PM₂.₅; captures 1987 PM10 NAAQS era |
| PM₂.₅ FRM/FEM | 88101 | **1999** | Primary aerosol burden metric |
| NO₂ | 42602 | **1999** | NOₓ proxy — secondary aerosol precursor |

**Runtime:** ~90–120 min (~1 400 API calls at 5 s/call). Fully resumable via per-state cache.

**Outputs:** `data/processed/{param}_annual_{start}_{end}_eastern_us.csv`

Run `02_eastern_us_trend_analysis.ipynb` after this completes.

In [ ]:
import os
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path

# ── AQS API Credentials ──────────────────────────────────────────────────────
# AQS_EMAIL = 
# API_KEY   = 
BASE_URL  = 'https://aqs.epa.gov/data/api/'

# ── Eastern US States (FIPS codes) ───────────────────────────────────────────
EASTERN_STATES = {
    'AL': '01', 'AR': '05', 'CT': '09', 'DC': '11', 'DE': '10',
    'FL': '12', 'GA': '13', 'IL': '17', 'IN': '18', 'KY': '21',
    'LA': '22', 'ME': '23', 'MD': '24', 'MA': '25', 'MI': '26',
    'MS': '28', 'MO': '29', 'NH': '33', 'NJ': '34', 'NY': '36',
    'NC': '37', 'OH': '39', 'PA': '42', 'RI': '44', 'SC': '45',
    'TN': '47', 'VT': '50', 'VA': '51', 'WV': '54', 'WI': '55',
}

# ── Pollutant Configuration ───────────────────────────────────────────────────
POLLUTANT_CONFIG = {
    'SO2':   {'code': '42401', 'start_year': 1971, 'end_year': 2024},
    'PM10':  {'code': '81102', 'start_year': 1987, 'end_year': 2024},
    'PM2.5': {'code': '88101', 'start_year': 1999, 'end_year': 2024},
    'NO2':   {'code': '42602', 'start_year': 1999, 'end_year': 2024},
}

YEAR_CHUNK = 1  

# ── Output Directories ────────────────────────────────────────────────────────
RAW_DIR       = Path('data/raw/annual')
PROCESSED_DIR = Path('data/processed')
FIGURES_DIR   = Path('figures')
for d in [RAW_DIR, PROCESSED_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
def aqs_get(endpoint, params, max_retries=3, retry_delay=30):
    """Query AQS REST API. Returns a DataFrame or None on failure."""
    p = {**params, 'email': AQS_EMAIL, 'key': API_KEY}
    url = BASE_URL + endpoint
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params=p, timeout=600)
            resp.raise_for_status()
            data   = resp.json()
            header = data.get('Header', [{}])[0]
            status = header.get('status', 'unknown')
            if status != 'Success':
                print(f'    API: {status} — {header.get("explanationOfStatus", "")}')
                return None
            body = data.get('Data', [])
            return pd.DataFrame(body) if body else pd.DataFrame()
        except requests.exceptions.Timeout:
            wait = retry_delay * (attempt + 1)
            print(f'    Timeout (attempt {attempt+1}/{max_retries}), waiting {wait}s...')
            if attempt < max_retries - 1:
                time.sleep(wait)
        except requests.exceptions.HTTPError as e:
            if resp.status_code == 429:
                print(f'    Rate limited, waiting {retry_delay}s...')
                time.sleep(retry_delay)
            else:
                print(f'    HTTP error: {e}')
                return None
        except Exception as e:
            print(f'    Error: {e}')
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
    return None


def year_chunks(start, end, chunk=YEAR_CHUNK):
    """Yield (bdate, edate) strings for chunked multi-year AQS queries."""
    yr = start
    while yr <= end:
        yr_end = min(yr + chunk - 1, end)
        yield f'{yr}0101', f'{yr_end}1231'
        yr = yr_end + 1


def download_annual_state(state_abbr, state_code, param_name, param_code,
                          start_year, end_year):
    """
    Download annual summary data for one state and pollutant.
    Cache filename includes start_year so different year ranges don't collide.
    """
    cache = RAW_DIR / f'annual_{param_name}_{state_abbr}_{start_year}_{end_year}.csv'
    if cache.exists():
        df = pd.read_csv(cache, dtype={'state_code': str, 'county_code': str, 'site_number': str})
        print(f'  {state_abbr} [{param_name}]: cache ({len(df):,} rows)')
        return df

    frames = []
    for bdate, edate in year_chunks(start_year, end_year):
        label = f'{bdate[:4]}-{edate[:4]}'
        print(f'  {state_abbr} [{param_name}] {label}...', end=' ', flush=True)
        df = aqs_get('annualData/byState', {
            'param': param_code,
            'bdate': bdate,
            'edate': edate,
            'state': state_code,
        })
        if df is not None and not df.empty:
            frames.append(df)
            print(f'{len(df):,} records')
        else:
            print('no data')
        time.sleep(5)

    if not frames:
        return pd.DataFrame()
    result = pd.concat(frames, ignore_index=True)
    result['state_abbr'] = state_abbr
    result.to_csv(cache, index=False)
    return result


print('Helper functions defined.')

In [ ]:
# ── Main Download Loop ────────────────────────────────────────────────────────

all_data = {}

for param_name, cfg in POLLUTANT_CONFIG.items():
    param_code = cfg['code']
    start_year = cfg['start_year']
    end_year   = cfg['end_year']
    out_file   = PROCESSED_DIR / f'{param_name}_annual_{start_year}_{end_year}_eastern_us.csv'

    print(f'\n{"="*65}')
    print(f'  {param_name}  (AQS param {param_code})  [{start_year}–{end_year}]')
    print(f'{"="*65}')

    if out_file.exists():
        print(f'  Combined file exists — loading from cache.')
        all_data[param_name] = pd.read_csv(out_file, low_memory=False)
        print(f'  {len(all_data[param_name]):,} records loaded.')
        continue

    state_frames = []
    for state_abbr, state_code in EASTERN_STATES.items():
        df = download_annual_state(state_abbr, state_code, param_name, param_code,
                                   start_year, end_year)
        if not df.empty:
            state_frames.append(df)

    if state_frames:
        combined = pd.concat(state_frames, ignore_index=True)
        combined.to_csv(out_file, index=False)
        all_data[param_name] = combined
        print(f'\n  Saved {len(combined):,} records -> {out_file.name}')
    else:
        print(f'\n  No data found for {param_name}.')


In [ ]:
# ── Download Summary ──────────────────────────────────────────────────────────

for param_name, df in all_data.items():
    if df.empty:
        print(f'{param_name}: no data')
        continue

    if 'year' in df.columns:
        yr_col = 'year'
    else:
        for c in ['date_local', 'Date']:
            if c in df.columns:
                df['year'] = pd.to_datetime(df[c]).dt.year
                yr_col = 'year'
                break

    yr_counts = df.groupby(yr_col).size()
    n_sites   = df.groupby(['state_code', 'county_code', 'site_number']).ngroups \
                if all(c in df.columns for c in ['state_code','county_code','site_number']) \
                else 'unknown'

    units = df['units_of_measure'].mode()[0] if 'units_of_measure' in df.columns else ''
    mean_range = ''
    if 'arithmetic_mean' in df.columns:
        vals = pd.to_numeric(df['arithmetic_mean'], errors='coerce').dropna()
        mean_range = f'{vals.min():.3f} – {vals.max():.3f} {units}'

    cfg = POLLUTANT_CONFIG[param_name]
    print(f'\n=== {param_name} ({cfg["start_year"]}–{cfg["end_year"]}) ===')
    print(f'  Total records : {len(df):,}')
    print(f'  Unique sites  : {n_sites}')
    print(f'  Year range    : {yr_counts.index.min()} – {yr_counts.index.max()}')
    print(f'  Records/year  : {yr_counts.mean():.0f} avg ({yr_counts.min()} – {yr_counts.max()})')
    print(f'  Annual mean   : {mean_range}')
    if 'state_abbr' in df.columns:
        print(f'  States        : {sorted(df["state_abbr"].dropna().unique())}')

In [ ]:
# PM2.5 Sulfate Speciation (CSN Network, param 88403)
#
# Set DOWNLOAD_SPECIATION = True to enable.

DOWNLOAD_SPECIATION = True

if DOWNLOAD_SPECIATION:
    spec_cfg = {'code': '88403', 'start_year': 1999, 'end_year': 2024}
    param_name = 'PM25_SO4'
    out_file = PROCESSED_DIR / f'{param_name}_annual_{spec_cfg["start_year"]}_{spec_cfg["end_year"]}_eastern_us.csv'

    print(f'Downloading PM2.5 sulfate speciation (param {spec_cfg["code"]})...')
    if out_file.exists():
        print(f'Already exists: {out_file}')
    else:
        state_frames = []
        for state_abbr, state_code in EASTERN_STATES.items():
            df = download_annual_state(state_abbr, state_code, param_name,
                                       spec_cfg['code'],
                                       spec_cfg['start_year'], spec_cfg['end_year'])
            if not df.empty:
                state_frames.append(df)
        if state_frames:
            combined = pd.concat(state_frames, ignore_index=True)
            combined.to_csv(out_file, index=False)
            print(f'Saved {len(combined):,} records -> {out_file.name}')
        else:
            print('No speciation data found.')
else:
    print('Speciation download skipped (DOWNLOAD_SPECIATION = False).')